# 08 — Estrazione delle feature di Whisper ed ECAPA-TDNN (no finetuning)

Estrae, per una partizione alla volta di ASVspoof 2019 LA:

- **Whisper-base, encoder, ultimo layer, livello di frame:** sequenza $(T, 512)$ in **float32**,
  con $T$ pari ai soli frame corrispondenti all'audio reale (un frame ogni 20 ms);
- **ECAPA-TDNN (`speechbrain/spkrec-ecapa-voxceleb`), livello di file:** embedding da 192 valori;
- **Whisper a livello di file:** media dei soli frame reali, calcolata durante l'estrazione e salvata a parte
  

**Una esecuzione per partizione** (`SPLIT = "train"` e poi `SPLIT = "dev"`): ogni output resta sotto i 9 GB,
contro i 17,8 GB di un'estrazione unica.

I modelli sono **congelati**: nessun addestramento, nessun gradiente.
L'audio è letto con `soundfile`, lo stesso decoder usato per le feature prosodiche, e passato
identico ai due modelli.


In [1]:
import os, re, json, time, math, sys, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import torch

SPLIT = os.environ.get("SPLIT", "train")        # "train" oppure "dev"
INPUT_ROOT = Path(os.environ.get("INPUT_ROOT", "/kaggle/input"))
LA_ROOT = None                                   # None = ricerca automatica
OUT_DIR = Path(os.environ.get("OUT_DIR", f"/kaggle/working/ptm_features_{SPLIT}"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

WHISPER_MODEL = "openai/whisper-base"
ECAPA_MODEL = "speechbrain/spkrec-ecapa-voxceleb"
SR = 16000
HOP_SAMPLES = 320                                # 20 ms per frame dell'encoder
MAX_FRAMES = 1500                                # 30 s, finestra dell'encoder
WHISPER_BATCH = int(os.environ.get("WHISPER_BATCH", 16))
ECAPA_BATCH = int(os.environ.get("ECAPA_BATCH", 16))
SHARD_FILES = int(os.environ.get("SHARD_FILES", 2000))
EXPECTED_ROWS = {"train": 25380, "dev": 24844}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"SPLIT = {SPLIT} | device = {DEVICE} | torch {torch.__version__}")

SPLIT = train | device = cuda | torch 2.10.0+cu128


In [2]:
def find_la_root(root):
    for dirpath, dirnames, _ in os.walk(root):
        dirnames[:] = [d for d in dirnames if d != "flac"]
        if "ASVspoof2019_LA_cm_protocols" in dirnames:
            return Path(dirpath)
    raise RuntimeError("Cartella con ASVspoof2019_LA_cm_protocols non trovata: imposta LA_ROOT.")

LA_ROOT = Path(LA_ROOT) if LA_ROOT else find_la_root(INPUT_ROOT)
pat = {"train": r"cm[._]train[._]trn", "dev": r"cm[._]dev[._]trl"}[SPLIT]
proto = [p for p in (LA_ROOT / "ASVspoof2019_LA_cm_protocols").iterdir() if re.search(pat, p.name)]
assert len(proto) == 1, proto
files = pd.read_csv(proto[0], sep=r"\s+", header=None, dtype=str, keep_default_na=False, engine="python",
                    names=["speaker", "utt_id", "unused", "attack", "label"]).drop(columns="unused")
files["attack"] = files["attack"].replace("-", "bonafide")
files["y_bonafide"] = (files["label"] == "bonafide").astype(np.int8)
files["path"] = [str(LA_ROOT / f"ASVspoof2019_LA_{SPLIT}" / "flac" / f"{u}.flac") for u in files["utt_id"]]

exp = EXPECTED_ROWS[SPLIT]
print(f"LA_ROOT = {LA_ROOT}\n{SPLIT}: {len(files)} file (attesi {exp})" + ("" if len(files) == exp else "  <-- ATTENZIONE"))
print(files["label"].value_counts().to_dict())
missing = [p for p in files["path"] if not os.path.exists(p)]
assert not missing, missing[:5]

LA_ROOT = /kaggle/input/datasets/angelopaldino/asvspoof2019-train-dev/LA
train: 25380 file (attesi 25380)
{'spoof': 22800, 'bonafide': 2580}


In [3]:
try:
    from transformers import WhisperModel, WhisperFeatureExtractor
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers"], check=True)
    from transformers import WhisperModel, WhisperFeatureExtractor
try:
    from speechbrain.inference.speaker import EncoderClassifier
except ImportError:
    try:
        from speechbrain.pretrained import EncoderClassifier
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "speechbrain"], check=True)
        from speechbrain.inference.speaker import EncoderClassifier

feat_extractor = WhisperFeatureExtractor.from_pretrained(WHISPER_MODEL)
whisper = WhisperModel.from_pretrained(WHISPER_MODEL).encoder.to(DEVICE).eval()
for p in whisper.parameters():
    p.requires_grad_(False)

ecapa = EncoderClassifier.from_hparams(source=ECAPA_MODEL, savedir=str(OUT_DIR / "_ecapa"),
                                       run_opts={"device": DEVICE})
ecapa.eval()
for p in ecapa.mods.parameters():
    p.requires_grad_(False)

print(f"Whisper: {sum(p.numel() for p in whisper.parameters()):,} parametri (encoder) | "
      f"d_model {whisper.config.d_model} | mel {feat_extractor.feature_size} bande | "
      f"chunk {feat_extractor.chunk_length}s | hop {feat_extractor.hop_length}")
print(f"ECAPA: {sum(p.numel() for p in ecapa.mods.parameters()):,} parametri | moduli: {list(ecapa.mods.keys())}")
print("ECAPA, chiavi degli hparams:", [k for k in ecapa.hparams.__dict__ if not k.startswith('_')][:20])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 41.9 MB/s eta 0:00:00


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

...

mean_var_norm_emb.ckpt:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

classifier.ckpt:   0%|          | 0.00/5.53M [00:00<?, ?B/s]

label_encoder.txt: 0.00B [00:00, ?B/s]

Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


Whisper: 20,590,592 parametri (encoder) | d_model 512 | mel 80 bande | chunk 30s | hop 160
ECAPA: 22,150,912 parametri | moduli: ['compute_features', 'mean_var_norm', 'embedding_model', 'mean_var_norm_emb', 'classifier']
ECAPA, chiavi degli hparams: ['n_mels', 'pretrained_path', 'out_n_neurons', 'compute_features', 'mean_var_norm', 'embedding_model', 'classifier', 'mean_var_norm_emb', 'modules', 'label_encoder', 'pretrainer', 'savedir']


In [4]:
def load_audio(path):
    x, sr = sf.read(path, dtype="float32", always_2d=False)
    assert sr == SR and x.ndim == 1, (path, sr, x.shape)
    return x

def n_frames_of(n_samples):
    return int(min(math.ceil(n_samples / HOP_SAMPLES), MAX_FRAMES))

@torch.no_grad()
def whisper_batch(waves):
    # -> lista di array (T, 512) float32, uno per file
    feats = feat_extractor([w for w in waves], sampling_rate=SR, return_tensors="pt")
    out = whisper(feats.input_features.to(DEVICE)).last_hidden_state.float().cpu().numpy()
    return [out[i, :n_frames_of(len(w))] for i, w in enumerate(waves)]

@torch.no_grad()
def ecapa_batch(waves):
    # -> array (B, 192) float32; wav_lens = lunghezze relative, per escludere il padding dal pooling
    lens = torch.tensor([len(w) for w in waves], dtype=torch.float32)
    maxlen = int(lens.max())
    batch = torch.zeros(len(waves), maxlen)
    for i, w in enumerate(waves):
        batch[i, :len(w)] = torch.from_numpy(w)
    emb = ecapa.encode_batch(batch.to(DEVICE), (lens / maxlen).to(DEVICE))
    return emb.squeeze(1).float().cpu().numpy()

In [5]:
def shard_path(k):
    return OUT_DIR / f"whisper_{SPLIT}_shard{k:03d}.npz"

n_shards = math.ceil(len(files) / SHARD_FILES)
t_start = time.perf_counter()
for k in range(n_shards):
    if shard_path(k).exists():
        continue
    chunk = files.iloc[k * SHARD_FILES:(k + 1) * SHARD_FILES]
    t0 = time.perf_counter()
    seqs, embs = [], []
    for i in range(0, len(chunk), WHISPER_BATCH):
        waves = [load_audio(p) for p in chunk["path"].iloc[i:i + WHISPER_BATCH]]
        seqs += whisper_batch(waves)
        for j in range(0, len(waves), ECAPA_BATCH):
            embs.append(ecapa_batch(waves[j:j + ECAPA_BATCH]))
    lens = np.array([len(s) for s in seqs], dtype=np.int64)
    np.savez(shard_path(k),
             X=np.concatenate(seqs).astype(np.float32),
             offsets=np.concatenate([[0], np.cumsum(lens)]).astype(np.int64),
             utt_id=chunk["utt_id"].to_numpy(),
             whisper_mean=np.stack([s.mean(0) for s in seqs]).astype(np.float32),
             ecapa=np.concatenate(embs).astype(np.float32))
    dt = time.perf_counter() - t0
    print(f"blocco {k + 1}/{n_shards}: {len(chunk)} file in {dt / 60:.1f} min "
          f"({dt / len(chunk) * 1000:.0f} ms/file) | {shard_path(k).stat().st_size / 1e9:.2f} GB "
          f"| residuo stimato {dt * (n_shards - k - 1) / 60:.0f} min")
print(f"Estrazione terminata in {(time.perf_counter() - t_start) / 60:.1f} min")

blocco 1/13: 2000 file in 2.0 min (60 ms/file) | 0.69 GB | residuo stimato 24 min
blocco 2/13: 2000 file in 2.0 min (60 ms/file) | 0.64 GB | residuo stimato 22 min
blocco 3/13: 2000 file in 2.0 min (61 ms/file) | 0.55 GB | residuo stimato 20 min
blocco 4/13: 2000 file in 2.2 min (67 ms/file) | 0.83 GB | residuo stimato 20 min
blocco 5/13: 2000 file in 2.2 min (67 ms/file) | 0.84 GB | residuo stimato 18 min
blocco 6/13: 2000 file in 2.2 min (66 ms/file) | 0.72 GB | residuo stimato 15 min
blocco 7/13: 2000 file in 2.2 min (65 ms/file) | 0.67 GB | residuo stimato 13 min
blocco 8/13: 2000 file in 2.2 min (66 ms/file) | 0.67 GB | residuo stimato 11 min
blocco 9/13: 2000 file in 2.2 min (65 ms/file) | 0.65 GB | residuo stimato 9 min
blocco 10/13: 2000 file in 2.1 min (64 ms/file) | 0.77 GB | residuo stimato 6 min
blocco 11/13: 2000 file in 2.1 min (63 ms/file) | 0.73 GB | residuo stimato 4 min
blocco 12/13: 2000 file in 2.1 min (64 ms/file) | 0.76 GB | residuo stimato 2 min
blocco 13/13: 138

In [6]:
shards = sorted(OUT_DIR.glob(f"whisper_{SPLIT}_shard*.npz"))
utt, lens, means, ecapas, bad = [], [], [], [], []
for p in shards:
    z = np.load(p, allow_pickle=True)
    utt.append(z["utt_id"]); lens.append(np.diff(z["offsets"]))
    means.append(z["whisper_mean"]); ecapas.append(z["ecapa"])
    if not np.isfinite(z["X"]).all() or not np.isfinite(z["ecapa"]).all():
        bad.append(p.name)
utt = np.concatenate(utt); lens = np.concatenate(lens)
means = np.concatenate(means); ecapas = np.concatenate(ecapas)

np.savez(OUT_DIR / f"whisper_mean_{SPLIT}.npz", whisper_mean=means, utt_id=utt)
np.savez(OUT_DIR / f"ecapa_{SPLIT}.npz", ecapa=ecapas, utt_id=utt)

index = files.drop(columns="path").copy()
index["n_frames"] = pd.Series(lens, index=pd.Index(utt)).reindex(index["utt_id"]).to_numpy()
index["shard"] = np.repeat(np.arange(len(shards)), [len(np.load(p, allow_pickle=True)["utt_id"]) for p in shards])
index.to_csv(OUT_DIR / f"index_{SPLIT}.csv", index=False)

durations = np.array([sf.info(p).frames for p in files["path"]])
expected = np.minimum(np.ceil(durations / HOP_SAMPLES), MAX_FRAMES)
print("File elaborati:", len(utt), "| attesi:", len(files))
print("Ordine identico al protocollo:", bool((utt == files['utt_id'].to_numpy()).all()))
print("Numero di frame coerente con la durata:", bool((index['n_frames'].to_numpy() == expected).all()))
print("Blocchi con valori non finiti:", bad if bad else "nessuno")
print("Embedding ECAPA:", ecapas.shape, "| norma media %.3f" % np.linalg.norm(ecapas, axis=1).mean())
print("Whisper, media per file:", means.shape)
print("Spazio totale: %.2f GB" % (sum(p.stat().st_size for p in OUT_DIR.glob('*.npz')) / 1e9))

File elaborati: 25380 | attesi: 25380
Ordine identico al protocollo: True
Numero di frame coerente con la durata: True
Blocchi con valori non finiti: nessuno
Embedding ECAPA: (25380, 192) | norma media 330.284
Whisper, media per file: (25380, 512)
Spazio totale: 9.07 GB


In [7]:
meta = {"split": SPLIT, "n_files": int(len(utt)), "whisper_model": WHISPER_MODEL, "ecapa_model": ECAPA_MODEL,
        "whisper_layer": "ultimo stato nascosto dell'encoder", "whisper_dtype": "float32",
        "whisper_level": "frame (20 ms), solo frame reali", "ecapa_level": "file (192)",
        "frozen": True, "sample_rate": SR, "hop_samples": HOP_SAMPLES, "max_frames": MAX_FRAMES,
        "shard_files": SHARD_FILES, "n_shards": len(shards),
        "format": "whisper_{split}_shardNNN.npz: X (frame concatenati, float32), offsets (N+1), utt_id, "
                  "whisper_mean (N,512), ecapa (N,192); piu' whisper_mean_{split}.npz, ecapa_{split}.npz, index_{split}.csv",
        "torch": torch.__version__, "device": DEVICE}
_ = (OUT_DIR / f"meta_{SPLIT}.json").write_text(json.dumps(meta, indent=2))
print("Output:", sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))
print("\nRicorda: salva la versione, poi rilancia il notebook con SPLIT = \"dev\".")

Output: ['ecapa_train.npz', 'index_train.csv', 'meta_train.json', 'whisper_mean_train.npz', 'whisper_train_shard000.npz', 'whisper_train_shard001.npz', 'whisper_train_shard002.npz', 'whisper_train_shard003.npz', 'whisper_train_shard004.npz', 'whisper_train_shard005.npz', 'whisper_train_shard006.npz', 'whisper_train_shard007.npz', 'whisper_train_shard008.npz', 'whisper_train_shard009.npz', 'whisper_train_shard010.npz', 'whisper_train_shard011.npz', 'whisper_train_shard012.npz']

Ricorda: salva la versione, poi rilancia il notebook con SPLIT = "dev".
